# Data Analysis · Week 9
## Accumulators, flags and nested loops

**TIA502 · School of Business · Instructor David Escobar-Castillejos**

Three patterns that solve nearly everything done inside a loop. They are not three topics: they
are three forms of the same structure, and recognising which one the question is asking for is
half of solving it.

It is worth saying where this goes. **Everything from today comes back in week 15 as a single line
of pandas.** The accumulator becomes `.sum()`, the counter becomes `.count()`, the flag becomes
`.any()`, and the nested loop becomes `groupby` on two columns. Practising it by hand now is what
later lets you trust the short line.

By the end of this notebook you will be able to:

1. Write an accumulator, a counter and a flag, and say what question each answers.
2. Interrupt or skip a pass with `break` and `continue`.
3. Use the `for` loop's `else`, which almost nobody knows about.
4. Read a nested loop and predict how many passes it makes before running it.

### How to use this notebook

Run the cells in order. Three fail on purpose or give a deliberately surprising result.

The data is the four campaigns from last week.

In [ ]:
campaigns = ["Instagram", "Meta", "Google", "TikTok"]
clicks = [5074, 3820, 6910, 1240]
spend = [38500, 29800, 51200, 9600]
conversions = [173, 118, 241, 39]

print(f"{'Campaign':<12}{'Clicks':>8}{'Spend':>12}{'CPC':>8}")
for i in range(len(campaigns)):
    print(f"{campaigns[i]:<12}{clicks[i]:>8,}{spend[i]:>12,}{spend[i] / clicks[i]:>8.2f}")

---
# Block 1 · The three patterns

Accumulate, count and mark.

| Pattern | The question | Starting value | Inside the loop |
|---|---|---|---|
| Accumulator | How much do they add up to? | `0` | `total += value` |
| Counter | How many qualify? | `0` | `if condition: n += 1` |
| Flag | Is there at least one? | `False` | `if condition: found = True` |

All three share the same structure. **A variable is declared before the loop**, with a starting
value meaning "nothing yet". **Inside the loop it gets updated** on every pass. **When it ends**,
that variable holds the answer.

In [ ]:
total_spend = 0
large_campaigns = 0
any_expensive = False

for i in range(len(campaigns)):
    total_spend += spend[i]

    if clicks[i] > 5000:
        large_campaigns += 1

    if spend[i] / clicks[i] > 7.75:
        any_expensive = True

print(f"Total spend:      ${total_spend:,}")
print(f"Large campaigns:  {large_campaigns}")
print(f"Any expensive:    {any_expensive}")

One pass, three answers.

**All three variables are declared before the `for`**, which is why they survive every pass.

**The accumulator** adds how much. It answers a question of magnitude.

**The counter** adds one. It answers a question of how many, not how much.

**The flag** only goes from `False` to `True`, and never comes back. It answers whether any exist.

## Counting and summing are not the same

This is the reading error that turns up most in an exam. Read it carefully: "how many campaigns
beat the target" and "how much do the campaigns that beat the target add up to" are two questions.

In [ ]:
TARGET = 0.03

how_many = 0
how_much = 0

for i in range(len(campaigns)):
    conversion = conversions[i] / clicks[i]
    if conversion >= TARGET:
        how_many += 1
        how_much += spend[i]

print(f"How many beat the target?      {how_many}")
print(f"How much did they add up to?  ${how_much:,}")

Three campaigns and 119,500 pesos. Both numbers come from the same pass and answer different
things.

The signal in the wording: **"how many" is a counter, "how much" is an accumulator.** It is almost
always in the first word of the question.

## The classic error: declaring it inside

**Predict before you run.** What is `total` when it finishes?

- **A.** 600, because it adds all three.
- **B.** 300, because it resets on every pass.
- **C.** 0, because the total is declared at the end.
- **D.** An error, because `total` does not exist before the loop.

In [ ]:
# FAILS ON PURPOSE. The accumulator sits inside the loop.
sales_list = [100, 200, 300]

for v in sales_list:
    total = 0
    total += v

print(total)

The answer is **B**, 300. Every pass wipes the total and starts again, so at the end it holds the
last record.

The program runs, raises nothing, and the wrong result looks perfectly normal. Three hundred is a
believable number for the sum of three sales.

**Indentation is what decides which is which.** Compare the two versions.

In [ ]:
print("With the accumulator INSIDE:")
for v in sales_list:
    total = 0
    total += v
    print(f"  pass with v={v}, total is now {total}")
print("  final:", total)

print()
print("With the accumulator OUTSIDE:")
total = 0
for v in sales_list:
    total += v
    print(f"  pass with v={v}, total is now {total}")
print("  final:", total)

The pass-by-pass trace makes it obvious. When a total does not add up, printing it inside the loop
is the fastest way to see where it goes.

## Accumulating the maximum and minimum

There is a fourth form of the same pattern, and it is the one for "the best" and "the worst".

In [ ]:
best_i = 0
worst_i = 0

for i in range(len(campaigns)):
    if spend[i] / clicks[i] < spend[best_i] / clicks[best_i]:
        best_i = i
    if spend[i] / clicks[i] > spend[worst_i] / clicks[worst_i]:
        worst_i = i

print(f"Best CPC:  {campaigns[best_i]:<12} ${spend[best_i] / clicks[best_i]:.2f}")
print(f"Worst CPC: {campaigns[worst_i]:<12} ${spend[worst_i] / clicks[worst_i]:.2f}")

The starting value is not zero: it is **the first element**. Starting at zero would break the
search for a minimum, because no cost per click is below zero and the answer would always be the
initial zero.

That is the "impossible starting value" error, and it turns up every time somebody initialises a
minimum at zero.

In [ ]:
# FAILS ON PURPOSE. Initialising a minimum at zero.
wrong_min = 0

for i in range(len(campaigns)):
    cpc = spend[i] / clicks[i]
    if cpc < wrong_min:
        wrong_min = cpc

print("Minimum starting at zero:", wrong_min, "<- no campaign is cheaper than free")

---
# Block 2 · Breaking the flow

Three statements that change a loop's normal path. Almost nobody knows the third.

| Statement | What it does | When it is used |
|---|---|---|
| `break` | Leaves the loop immediately | You already found what you were looking for and continuing is waste |
| `continue` | Skips to the next pass | This record does not apply and you do not want to nest a huge `if` |
| `for` loop `else` | Runs only if the loop finished without a `break` | To say "I walked it all and found nothing" |

In [ ]:
for i in range(len(campaigns)):
    if clicks[i] < 2000:
        continue

    if spend[i] / clicks[i] > 7.75:
        print(f"First expensive one: {campaigns[i]}")
        break
else:
    print("No campaign clears the threshold.")

**`continue`.** Campaigns with fewer than 2,000 clicks do not have enough volume to judge. They get
skipped without nesting an `if` around everything else.

**`break`.** As soon as it finds the first one, it leaves. Walking the rest would not change the
answer.

**The `else`.** It lines up with the `for`, not with the `if`. It runs only if the loop reached the
end without hitting a `break`.

The trace:

| `i` | Campaign | clicks | What it does |
|---|---|---|---|
| 0 | Instagram | 5074 | Passes the filter. 7.59 does not clear 7.75, carries on |
| 1 | Meta | 3820 | Passes the filter. 7.80 does clear it, prints and leaves |
| 2 | Google | 6910 | Not evaluated, the `break` already left |
| 3 | TikTok | 1240 | Not evaluated |

The `for` loop's `else` does not run, because the loop left through a `break`. That is exactly its
purpose.

Raise the threshold to see the other case.

In [ ]:
THRESHOLD = 20.00

for i in range(len(campaigns)):
    if clicks[i] < 2000:
        continue

    if spend[i] / clicks[i] > THRESHOLD:
        print(f"First expensive one: {campaigns[i]}")
        break
else:
    print(f"No campaign clears ${THRESHOLD:.2f} per click.")

Now the `else` did run, because the loop reached the end without a `break`.

Without that `else`, getting the same result would need a flag and an `if` after the loop. The
`for` loop's `else` is exactly that, packaged.

In [ ]:
# The same thing, with a flag. It works identically and takes two more lines.
found = False

for i in range(len(campaigns)):
    if clicks[i] < 2000:
        continue
    if spend[i] / clicks[i] > THRESHOLD:
        print(f"First expensive one: {campaigns[i]}")
        found = True
        break

if not found:
    print(f"No campaign clears ${THRESHOLD:.2f} per click.")

Both forms are correct. The flag version is understandable without knowing the `for` loop's `else`,
which is why many people prefer it. The other is shorter and cannot forget to update the flag.

## The risk of a misaligned `else`

In [ ]:
# FAILS ON PURPOSE. The else lines up with the if, not with the for.
for i in range(len(campaigns)):
    if spend[i] / clicks[i] > 7.75:
        print(f"Expensive: {campaigns[i]}")
    else:
        print(f"  (cheap: {campaigns[i]})")

That `else` belongs to the `if` and runs on every pass that fails, which is a completely different
thing. Both programs are valid and do different things, and all that separates them is four
spaces.

---
# Block 3 · Nested loops

A loop inside another. The inner one makes all of its passes for every pass of the outer one.

In [ ]:
regions = ["North", "Centre"]
channels = ["Retail", "Online"]

for region in regions:
    for channel in channels:
        print(f"{region} · {channel}")

**The multiplication.** Two regions by two channels give four passes. With four and three it would
be twelve.

**The order.** The outer one advances a position only when the inner one has finished all of its.

**The limit.** A hundred by a hundred is ten thousand passes. Nesting three levels over long lists
gets slow quickly.

Count them rather than assuming.

In [ ]:
regions = ["North", "Centre", "South", "West"]
channels = ["Retail", "Online", "Wholesale"]

passes = 0
for region in regions:
    for channel in channels:
        passes += 1

print(f"{len(regions)} regions × {len(channels)} channels = {passes} passes")

## A cross-tab with data

Nesting becomes useful when every combination produces a report row.

In [ ]:
# One figure per combination, written by hand for the example.
SALES = {
    ("North", "Retail"): 1331426, ("North", "Online"): 978286, ("North", "Wholesale"): 2042264,
    ("Centre", "Retail"): 490472, ("Centre", "Online"): 1291740, ("Centre", "Wholesale"): 2136767,
    ("South", "Retail"): 271090, ("South", "Online"): 420216, ("South", "Wholesale"): 861697,
    ("West", "Retail"): 738049, ("West", "Online"): 589081, ("West", "Wholesale"): 1702889,
}

print(f"{'Region':<10}" + "".join(f"{c:>12}" for c in channels) + f"{'Total':>12}")
print("-" * 58)

grand_total = 0
for region in regions:
    row = 0
    for channel in channels:
        row += SALES[(region, channel)]
    grand_total += row
    cells = "".join(f"{SALES[(region, channel)] / 1000:>12,.0f}" for channel in channels)
    print(f"{region:<10}{cells}{row / 1000:>12,.0f}")

print("-" * 58)
print(f"{'Total':<10}{'':>36}{grand_total / 1000:>12,.0f}")

One accumulator per row and another overall, inside a nesting. Those twenty lines are exactly what
week 15.3 writes like this:

```python
sales.pivot_table(index="region", columns="channel", values="amount",
                  aggfunc="sum", margins=True)
```

Worth seeing them side by side once. What pandas takes away is not the concept, it is the
bookkeeping.

## Reusing the inner loop's variable

In [ ]:
# FAILS ON PURPOSE. Both loops use i.
for i in range(3):
    for i in range(2):        # the inner one overwrites the outer
        pass
    print("Outer pass, i is:", i)

The outer `i` is lost: when the inner one finishes it is 1, always. With short lists the symptom is
subtle; with real indices, the walk falls apart silently.

**The two loops' variables have to be named differently**, and preferably say what they walk:
`region` and `channel` rather than `i` and `j`.

## Was nesting even needed?

Before putting a loop inside another, ask whether a `continue` in the first would have done the
same.

In [ ]:
# Unnecessary nesting: walk everything and filter inside.
print("With nesting:")
for region in regions:
    for channel in channels:
        if channel == "Wholesale":
            print(f"  {region} · {channel}: {SALES[(region, channel)] / 1000:,.0f}")

# The same thing, without the inner loop.
print()
print("Without nesting:")
for region in regions:
    print(f"  {region} · Wholesale: {SALES[(region, 'Wholesale')] / 1000:,.0f}")

Twelve passes against four, for the same result. The inner loop existed only to discard two out of
every three cases.

---
# Exercises

The solutions sit at the very bottom of the notebook.

## The three patterns

### Exercise 1 · One of each

In a single pass over the four campaigns, work out: the total clicks, how many have more than 150
conversions, and whether any has a conversion rate below 3 %.

### Exercise 2 · Counting against summing

Answer all four questions, carefully distinguishing counter from accumulator:

1. How many campaigns cost more than 7.60 per click?
2. How much was spent on those campaigns?
3. How many conversions did they bring between them?
4. What did each conversion cost, on average, in those campaigns?

### Exercise 3 · The best by another metric

Find the campaign with the best cost per conversion, not per click. Print its name, its cost per
conversion and how much better it is than the worst.

Initialise with the first element, not with zero.

## Breaking the flow

### Exercise 4 · `continue` that avoids a big `if`

Write a loop that reports the cost per conversion only for campaigns with more than 100
conversions, using `continue` to skip the rest.

Then write it with an `if` wrapping the whole body, and compare which reads better.

### Exercise 5 · `break` with `else`

Find the first campaign that satisfies two conditions at once: more than 5,000 clicks and a
conversion rate above 3.4 %. If there is none, say so with the `for` loop's `else`.

Test with a threshold that finds one and with one that does not.

### Exercise 6 · Counting without walking too far

Write a loop that stops as soon as accumulated spend passes 100,000, and prints how many campaigns
it took.

## Nesting

### Exercise 7 · Predicting the passes

Without running anything, say how many times something prints in each of these three, then check.

```python
for a in range(3):
    for b in range(4):
        print(a, b)

for a in range(3):
    for b in range(a):
        print(a, b)

for a in range(3):
    for b in range(4):
        if b == 2:
            break
        print(a, b)
```

The second and third are the interesting ones. Explain why in a comment.

### Exercise 8 · A cross-tab report

With two lists of categories from your field, for example region and product, write a nested loop
printing every combination with a computed metric. Add an accumulator and a counter to the pass.

The two loops' variables have to be named differently and say what they walk.

The test: count by hand how many lines it should print. If it does not match, the nesting is wrong.

---
## Three ideas to take away

**The variable lives outside the loop.** Declaring it inside resets it every pass, and the wrong
result looks perfectly normal.

**Counting and summing are not the same.** "How many qualify" is a counter, "how much do they add up
to" is an accumulator, and the wording almost always says which in its first word.

**The passes multiply.** Two by two is four, and a hundred by a hundred is ten thousand. That is
where nesting stops being free.

Next session is how to package a calculation so you never have to write it again.

---
# Solutions

### Exercise 1

```python
total_clicks = 0
many_conversions = 0
any_low = False

for i in range(len(campaigns)):
    total_clicks += clicks[i]
    if conversions[i] > 150:
        many_conversions += 1
    if conversions[i] / clicks[i] < 0.03:
        any_low = True

print(f"Total clicks:                {total_clicks:,}")
print(f"With more than 150 conversions: {many_conversions}")
print(f"Any below 3 %:               {any_low}")
```

One pass for three answers. Walking three times also works and costs three times as much, which with
four campaigns makes no difference and with three hundred thousand does.

### Exercise 2

```python
THRESHOLD = 7.60

how_many = 0
spent = 0
total_conversions = 0

for i in range(len(campaigns)):
    if spend[i] / clicks[i] > THRESHOLD:
        how_many += 1
        spent += spend[i]
        total_conversions += conversions[i]

print(f"1. How many cost more than {THRESHOLD}: {how_many}")
print(f"2. How much was spent:              ${spent:,}")
print(f"3. How many conversions they brought: {total_conversions}")
print(f"4. Average cost per conversion:     ${spent / total_conversions:,.2f}")
```

The fourth is the interesting one: it is the accumulated over the accumulated, not the average of
the averages. It is the same care as the overall cost per click from last week.

### Exercise 3

```python
best = 0
worst = 0

for i in range(len(campaigns)):
    cpa = spend[i] / conversions[i]
    if cpa < spend[best] / conversions[best]:
        best = i
    if cpa > spend[worst] / conversions[worst]:
        worst = i

cpa_best = spend[best] / conversions[best]
cpa_worst = spend[worst] / conversions[worst]

print(f"Best:  {campaigns[best]:<12} ${cpa_best:>8,.2f} per conversion")
print(f"Worst: {campaigns[worst]:<12} ${cpa_worst:>8,.2f} per conversion")
print(f"The best costs {cpa_worst / cpa_best:.2f} times less than the worst")
```

Initialising at `0` as an **index** is correct; initialising the cost at `0` as a **value** would not
be. The difference is that here zero is a valid position in the list, not an impossible value of the
metric.

### Exercise 4

```python
print("With continue:")
for i in range(len(campaigns)):
    if conversions[i] <= 100:
        continue
    print(f"  {campaigns[i]:<12} ${spend[i] / conversions[i]:>8,.2f}")

print("\nWith a wrapping if:")
for i in range(len(campaigns)):
    if conversions[i] > 100:
        print(f"  {campaigns[i]:<12} ${spend[i] / conversions[i]:>8,.2f}")
```

With a single line inside, the difference does not show. It shows when the body is fifteen lines:
`continue` keeps them all at one level of indentation, and the wrapping `if` pushes them all one
level deeper.

### Exercise 5

```python
for conv_threshold in [0.034, 0.10]:
    print(f"Looking for a conversion above {conv_threshold:.1%}:")
    for i in range(len(campaigns)):
        if clicks[i] <= 5000:
            continue
        if conversions[i] / clicks[i] > conv_threshold:
            print(f"  Found: {campaigns[i]}")
            break
    else:
        print("  None satisfies both conditions.")
    print()
```

At 3.4 % it finds Google; at 10 % it finds nothing and the `else` runs. Putting both thresholds in an
outer loop is what lets you test both paths without duplicating the code.

### Exercise 6

```python
LIMIT = 100000
accumulated = 0
how_many = 0

for i in range(len(campaigns)):
    accumulated += spend[i]
    how_many += 1
    if accumulated > LIMIT:
        break

print(f"It took {how_many} campaigns to pass {LIMIT:,}")
print(f"Accumulated on stopping: {accumulated:,}")
```

Three campaigns and 119,500. Note the counter goes up **before** the `break`, because the campaign
that crossed the limit does count. Putting it after would give two, and both readings need you to
decide which you meant.

### Exercise 7

```python
# The first: 12 times. Three by four, no conditions.
# The second: 3 times. range(a) gives zero passes when a is 0, one when it is 1
#   and two when it is 2. Zero plus one plus two is three.
# The third: 6 times. The break cuts the inner loop at b == 2, so every outer
#   pass prints b = 0 and b = 1, and there are three outer passes.

n = 0
for a in range(3):
    for b in range(4):
        n += 1
print("First:", n)

n = 0
for a in range(3):
    for b in range(a):
        n += 1
print("Second:", n)

n = 0
for a in range(3):
    for b in range(4):
        if b == 2:
            break
        n += 1
print("Third:", n)
```

The second is the "each against the earlier ones" pattern, and it shows up when you compare every
pair in a list without repeating. The third teaches that a `break` in the inner loop **only leaves
the inner one**, not both.

### Exercise 8

There is no published solution, because the categories differ for everyone. It is graded on four
things: that the two loops' variables have names saying what they walk, that the accumulator and
counter are declared outside both loops, that the hand count of rows matches the output, and that
the report has a header.